# CLIPasso + diffvg on Google Colab

本 notebook 会把仓库拉取到 `/content/clipasso-colab`，安装依赖、编译 diffvg、下载 U2Net 权重，然后对 `target_images/` 中的图片生成简笔画。

**使用前**：菜单 `代码执行程序 -> 更改运行时类型`，硬件加速器选择 **GPU**（T4 即可）。

运行顺序：依次执行下面的单元格即可。第一次运行编译 diffvg 需要几分钟。

In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/tuanzi091/clipasso-colab.git"
REPO_DIR = "/content/clipasso-colab"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

In [ ]:
%cd /content/clipasso-colab
!pip install -q -r requirements-colab.txt

## 1. 编译 diffvg

优先尝试 CUDA 版本；如果 Colab 的 CUDA/编译器版本不兼容，会自动回退到 CPU 版本（仍然可以正常出图，只是慢一些）。

In [ ]:
%cd /content/clipasso-colab
!python colab_setup.py --build-diffvg

## 2. 下载权重

- U2Net 前景分割权重：约 219MB，来自官方 Google Drive。
- CLIP 权重：首次运行时也会自动下载；这里可以提前下好（约 650MB）。

In [ ]:
%cd /content/clipasso-colab
!python colab_setup.py --download-weights

In [ ]:
%cd /content/clipasso-colab
!python colab_setup.py --download-clip

## 3. 环境自检

In [ ]:
%cd /content/clipasso-colab
!python colab_setup.py --verify

## 4. 生成简笔画

默认对 `target_images/camel.png` 运行。把 `camel.png` 换成 `target_images/` 下的其他文件名即可。

- `--num_paths`：动态模式下作为最大笔触数
- `--num_iter`：迭代次数，越大越精细
- `--use_gpu 1`：使用 GPU

输出目录为 `results/<名字>/`，真正的结果在 `results/<名字>/<名字>/` 子目录中（包含 `best_iter.jpg`、`best_iter.svg`、`final_svg.svg` 以及迭代过程图）。

In [ ]:
%cd /content/clipasso-colab
!python process_image.py target_images/camel.png --output_dir results/camel --num_paths 24 --num_iter 500 --use_gpu 1

## 5. 查看结果

In [ ]:
import glob
import os

from IPython.display import Image as IPImage, SVG, display

out_dir = "results/camel"
jpgs = sorted(glob.glob(os.path.join(out_dir, "**", "best_iter.jpg"), recursive=True))
svgs = sorted(glob.glob(os.path.join(out_dir, "**", "best_iter.svg"), recursive=True))

if jpgs:
    display(IPImage(filename=jpgs[0]))
if svgs:
    display(SVG(filename=svgs[0]))
if not jpgs and not svgs:
    print("没有找到输出文件，请检查上一格的运行日志。")

## 6. 打包下载

In [ ]:
import shutil

from google.colab import files

shutil.make_archive("/content/clipasso_results", "zip", "results")
files.download("/content/clipasso_results.zip")

## 备用：官方命令行入口

也可以用原始的 `run_object_sketching.py`（会跑多个随机种子并自动选最好的一个）：

```python
%cd /content/clipasso-colab
!python run_object_sketching.py --target_file camel.png --num_strokes 16 --num_iter 500 --num_sketches 1
```

## 常见问题

1. **diffvg 编译失败**：先确认 `requirements-colab.txt` 里的 `cmake==3.31.6`、`pybind11==2.13.6` 已安装（不要用 CMake 4.x）。仍有报错就把编译日志保存下来排查。
2. **CUDA 编译失败但 CPU 编译成功**：属于正常的自动回退，可以继续使用；速度会明显慢于 GPU。
3. **CLIP 权重下载失败**：在 Colab 中重跑 `--download-clip` 即可；国内网络环境建议使用 Colab 的默认网络（Google 域名可达）。
4. **OOM**：把 `--num_paths` 或 `--num_iter` 调小。